# 🌡️ Machine Learning Model: Predicting Today's Temperature Based on Yesterday's Temperature

**College Project Submission**

### Objective:
Predicting weather/temperature is a classic time-series regression task. In this project, we build an end-to-end Machine Learning pipeline that uses **yesterday's temperature** alongside seasonal lag features to predict **today's temperature**.

### Project Pipeline:
1. **Synthetic Data Generation**: 3 years of daily temperature data with real-world seasonal harmonics & autocorrelation.
2. **Exploratory Data Analysis (EDA)**: Visualizing temporal trends, correlation heatmaps, and scatter plots.
3. **Model Selection & Training**: Comparing multiple models (Linear Regression, Polynomial Regression, Random Forest, SVR, Neural Network).
4. **Evaluation & Comparison**: RMSE, MAE, R² metrics and performance plots.
5. **Real-time Prediction Interface**: Testing custom input predictions.

--- 
## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
print("✅ Libraries imported successfully!")

--- 
## 2. Generate Synthetic Weather Dataset

In [ ]:
np.random.seed(42)
num_days = 1095  # 3 years

dates = pd.date_range(start='2021-01-01', periods=num_days, freq='D')
day_of_year = dates.dayofyear.values
month = dates.month.values

# Annual sine wave pattern (base temp 25°C + 12°C variation)
base_temp = 25.0 + 12.0 * np.sin(2 * np.pi * (day_of_year - 80) / 365.25)

# Autocorrelated noise modeling real-world persistence
noise = np.zeros(num_days)
curr_noise = 0
for t in range(num_days):
    curr_noise = 0.75 * curr_noise + np.random.normal(0, 1.8)
    noise[t] = curr_noise

temp_today = base_temp + noise

df = pd.DataFrame({
    'date': dates,
    'temp_today': np.round(temp_today, 2),
    'day_of_year': day_of_year,
    'month': month
})

# Create Lag Features
df['temp_yesterday'] = df['temp_today'].shift(1)
df['temp_7day_avg'] = df['temp_today'].shift(1).rolling(window=7).mean()

# Drop NaNs from shift
df = df.dropna().reset_index(drop=True)

print(f"Dataset Shape: {df.shape}")
df.head()

--- 
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Summary Statistics
df[['temp_today', 'temp_yesterday', 'temp_7day_avg']].describe().round(2)

In [ ]:
# Visualizing Temperature over 3 Years
plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['temp_today'], label="Today's Temperature (°C)", alpha=0.75, color='#1f77b4')
plt.plot(df['date'], df['temp_7day_avg'], label="7-Day Moving Avg (°C)", color='#ff7f0e', linewidth=2)
plt.title("Daily Temperature Trend Across 3 Years")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.show()

In [ ]:
# Relationship between Yesterday's and Today's Temperature
plt.figure(figsize=(7, 6))
sns.regplot(x='temp_yesterday', y='temp_today', data=df, scatter_kws={'alpha':0.3, 'color':'green'}, line_kws={'color':'red'})
plt.title("Yesterday's vs Today's Temperature Correlation")
plt.xlabel("Yesterday's Temperature (°C)")
plt.ylabel("Today's Temperature (°C)")
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.drop(columns=['date']).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

--- 
## 4. Model Training & Evaluation

In [ ]:
X = df[['temp_yesterday', 'day_of_year', 'month', 'temp_7day_avg']]
y = df['temp_today']

# 80% Train, 20% Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Linear Regression': LinearRegression(),
    'Polynomial Regression (Deg 2)': make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor (SVR)': SVR(kernel='rbf', C=10.0, epsilon=0.1),
    'Neural Network (MLP)': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

results = {}
preds = {}

for name, model in models.items():
    if name in ['Support Vector Regressor (SVR)', 'Neural Network (MLP)']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE (°C)': round(rmse, 3), 'MAE (°C)': round(mae, 3), 'R² Score': round(r2, 4)}
    preds[name] = y_pred

results_df = pd.DataFrame(results).T
results_df.sort_values(by='R² Score', ascending=False)

--- 
## 5. Visualizing Model Performance

In [ ]:
# Model Performance Barplot
plt.figure(figsize=(10, 5))
sns.barplot(x=results_df.index, y=results_df['RMSE (°C)'], hue=results_df.index, palette='Blues_d', legend=False)
plt.title("Model Comparison: Root Mean Squared Error (Lower is better)")
plt.xticks(rotation=15, ha='right')
plt.show()

In [ ]:
# Test Set Predictions vs Actual
plt.figure(figsize=(14, 6))
test_dates = df.iloc[X_test.index]['date']
plt.plot(test_dates, y_test.values, label='Actual Temp', color='black', alpha=0.8, linewidth=1.5)

for name, y_pred in preds.items():
    plt.plot(test_dates, y_pred, label=name, alpha=0.7, linestyle='--')

plt.title("Model Predictions vs Actual Temperatures (Test Set)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.show()

--- 
## 6. Make Custom Predictions

In [ ]:
# Demo: Predict today's temperature when yesterday was 28.5°C on July 15th (day 196)
sample_temp_yesterday = 28.5
sample_day_of_year = 196
sample_month = 7
sample_7day_avg = 28.0

sample_df = pd.DataFrame([{
    'temp_yesterday': sample_temp_yesterday,
    'day_of_year': sample_day_of_year,
    'month': sample_month,
    'temp_7day_avg': sample_7day_avg
}])

best_model = models['Random Forest Regressor']
pred = best_model.predict(sample_df)[0]

print(f"Yesterday's Temperature: {sample_temp_yesterday}°C")
print(f"📌 Predicted Today's Temperature: {pred:.2f}°C")